# 1. Primera prueba de SAM3

Si lo quieren correr necesitan un token de hugging face para leer. 

Se llama **USER ACCESS TOKEN** y se necesita en modo lectura. Lo puse en un JSON llamado API.json junto con la libreta.

Github no me deja pasar el mio :'(.

In [1]:
%pip install -U "triton-windows<3.4"
%pip install torch torchvision torchaudio
%pip install transformers
%pip install accelerate
%pip install opencv-python
%pip install numpy
%pip install Pillow
%pip install tqdm

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
^C
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached numpy-2.4.6-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
    --------------------------------------- 0.8/40.2

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.4.6 which is incompatible.
sam3 0.1.0 requires numpy<2,>=1.26, but you have numpy 2.4.6 which is incompatible.
scipy 1.14.1 requires numpy<2.3,>=1.23.5, but you have numpy 2.4.6 which is incompatible.


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [7]:
import torch
import gc
from PIL import Image, ImageDraw
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
import matplotlib.pyplot as plt
from huggingface_hub import login
from contextlib import nullcontext
import json
import numpy as np

# --- 1. LIMPIEZA DE MEMORIA ---
torch.cuda.empty_cache()
gc.collect()

# --- 2. AUTENTICACIÓN Y CONFIGURACIÓN ---
with open("API.json", "r") as f:
    key = json.load(f)["API"]
login(token=key)

device = "cuda" if torch.cuda.is_available() else "cpu"
autocast_ctx = torch.autocast(device_type="cuda", dtype=torch.bfloat16) if device == "cuda" else nullcontext()

# --- 3. CARGA INTELIGENTE DEL MODELO ---
if 'sam' not in globals():
    print("Cargando el modelo por primera vez... (esto tomará un momento)")
    sam = build_sam3_image_model()
    sam.to(device)
    # ¡SECRETO REVELADO!: Bajamos el umbral aquí para obligar al modelo a aceptar detecciones
    processor = Sam3Processor(sam, confidence_threshold=0.15)
else:
    print("⚡ El modelo ya está en la memoria VRAM. Saltando carga.")
    # Nos aseguramos de aplicar el umbral bajo si el modelo ya estaba en memoria
    processor.confidence_threshold = 0.15

# --- 4. PREPARAR LA IMAGEN ---
imagen_path = r"C:\Users\Oscar\Documentos\FutBot\Prueba\pelotas.jpg"
imagen = Image.open(imagen_path).convert("RGB")

# --- 5. INFERENCIA SEGURA (CON PROMPT DE PRUEBA UNIVERSAL) ---
with torch.no_grad():
    with autocast_ctx:
        # Resetear prompts viejos por si acaso
        if 'inference_state' in globals():
            try:
                processor.reset_all_prompts(inference_state)
            except:
                pass
                
        inference_state = processor.set_image(imagen)
        
        # PROMPT UNIVERSAL: Busquemos "ball" de forma genérica para forzar una respuesta
        prompt = "soccer"
        output = processor.set_text_prompt(state=inference_state, prompt=prompt)

# Extraemos los datos usando .get() de forma segura
masks = output.get("masks")
boxes = output.get("boxes")
scores = output.get("scores")

# --- DEBUG INFO ---
print("\n--- DEBUG INFO ---")
if boxes is not None:
    print(f"Cajas encontradas (shape): {boxes.shape}")
    print(f"Confianzas obtenidas: {scores}")
else:
    print("Boxes es None. El modelo sigue sin encontrar nada.")
print("--------------------\n")

# --- 6. FUNCIÓN UNIFICADA DE DIBUJO (MÁSCARAS + CAJAS) ---
def visualizar_resultados(image, masks, boxes, scores):
    img_dibujada = image.copy()
    
    # Primero: Dibujamos las máscaras si existen
    if masks is not None and len(masks) > 0:
        img_dibujada = img_dibujada.convert("RGBA")
        
        # --- ARREGLO: Convertir de bfloat16 a float32 antes de numpy() ---
        masks_np = (masks.cpu().to(torch.float32).numpy() * 255).astype(np.uint8)
        
        # Extraemos máscaras 2D limpias sin dimensiones extra
        mascaras_2d = []
        for m in masks_np:
            m_sq = np.squeeze(m)
            if len(m_sq.shape) == 2:
                mascaras_2d.append(m_sq)
            elif len(m_sq.shape) == 3:
                mascaras_2d.append(m_sq[0])
                
        n_masks = len(mascaras_2d)
        cmap = plt.colormaps.get_cmap("rainbow").resampled(max(n_masks, 1))
        colors = [tuple(int(c * 255) for c in cmap(i)[:3]) for i in range(n_masks)]

        for mask_arr, color in zip(mascaras_2d, colors):
            mask_img = Image.fromarray(mask_arr, mode="L") 
            overlay = Image.new("RGBA", img_dibujada.size, color + (0,))
            alpha = mask_img.point(lambda v: int(v * 0.5))
            overlay.putalpha(alpha)
            img_dibujada = Image.alpha_composite(img_dibujada, overlay)
            
    # Segundo: Dibujamos las cajas sobre la imagen estilo Tesla
    img_dibujada = img_dibujada.convert("RGB") 
    draw = ImageDraw.Draw(img_dibujada)
    
    if boxes is not None and len(boxes) > 0 and boxes.shape[0] > 0:
        
        # Extraemos los tensores correctamente
        cajas_tensor = boxes[0] if len(boxes.shape) > 2 else boxes
        puntajes_tensor = scores[0] if len(scores.shape) > 1 else scores
        
        # --- ARREGLO: Convertir de bfloat16 a float32 antes de numpy() ---
        cajas = cajas_tensor.cpu().to(torch.float32).numpy()
        puntajes = puntajes_tensor.cpu().to(torch.float32).numpy()
        
        for caja, puntaje in zip(cajas, puntajes):
            # Dibujar rectángulo verde fluorescente
            draw.rectangle(caja.tolist(), outline="#00FF00", width=4) 
            
            # Fondo para el texto de confianza
            texto = f"Conf: {puntaje:.2f}"
            y_text = max(0, caja[1] - 20)
            draw.rectangle([caja[0], y_text, caja[0] + 75, y_text + 20], fill="#00FF00")
            
            # Texto negro centrado en el fondo
            draw.text((caja[0] + 2, y_text + 2), texto, fill="black")
    else:
        print("⚠️ No hay cajas detectadas para dibujar.")
            
    return img_dibujada

# --- 7. MOSTRAR RESULTADO FINAL ---
resultado_final = visualizar_resultados(imagen, masks, boxes, scores)
resultado_final.show()

Cargando el modelo por primera vez... (esto tomará un momento)


KeyboardInterrupt: 

In [ ]:
# Solo use esto para que funcionara este pedazo de basura, se quedaba cargado en mi tarjeta de video y no queria salirse

import torch
print(torch.cuda.get_device_name(0))

free, total = torch.cuda.mem_get_info()
print("Libre GB:", free / 1024**3)
print("Total GB:", total / 1024**3)

NVIDIA GeForce RTX 3060 Laptop GPU
Libre GB: 0.0
Total GB: 5.99951171875


# 2. Ver video con SAM3

Tengo todos los videos en una carpeta llamada Videos, use la función $load_all_videos$ para cargarlos todos en una lista y 

In [1]:
def visualizar_resultados(image, masks, boxes, scores):
    img_dibujada = image.copy()
    
    # Primero: Dibujamos las máscaras si existen
    if masks is not None and len(masks) > 0:
        img_dibujada = img_dibujada.convert("RGBA")
        
        # --- ARREGLO: Convertir de bfloat16 a float32 antes de numpy() ---
        masks_np = (masks.cpu().to(torch.float32).numpy() * 255).astype(np.uint8)
        
        # Extraemos máscaras 2D limpias sin dimensiones extra
        mascaras_2d = []
        for m in masks_np:
            m_sq = np.squeeze(m)
            if len(m_sq.shape) == 2:
                mascaras_2d.append(m_sq)
            elif len(m_sq.shape) == 3:
                mascaras_2d.append(m_sq[0])
                
        n_masks = len(mascaras_2d)
        cmap = plt.colormaps.get_cmap("rainbow").resampled(max(n_masks, 1))
        colors = [tuple(int(c * 255) for c in cmap(i)[:3]) for i in range(n_masks)]

        for mask_arr, color in zip(mascaras_2d, colors):
            mask_img = Image.fromarray(mask_arr, mode="L") 
            overlay = Image.new("RGBA", img_dibujada.size, color + (0,))
            alpha = mask_img.point(lambda v: int(v * 0.5))
            overlay.putalpha(alpha)
            img_dibujada = Image.alpha_composite(img_dibujada, overlay)
            
    # Segundo: Dibujamos las cajas sobre la imagen estilo Tesla
    img_dibujada = img_dibujada.convert("RGB") 
    draw = ImageDraw.Draw(img_dibujada)
    
    if boxes is not None and len(boxes) > 0 and boxes.shape[0] > 0:
        
        # Extraemos los tensores correctamente
        cajas_tensor = boxes[0] if len(boxes.shape) > 2 else boxes
        puntajes_tensor = scores[0] if len(scores.shape) > 1 else scores
        
        # --- ARREGLO: Convertir de bfloat16 a float32 antes de numpy() ---
        cajas = cajas_tensor.cpu().to(torch.float32).numpy()
        puntajes = puntajes_tensor.cpu().to(torch.float32).numpy()
        
        for caja, puntaje in zip(cajas, puntajes):
            # Dibujar rectángulo verde fluorescente
            draw.rectangle(caja.tolist(), outline="#00FF00", width=4) 
            
            # Fondo para el texto de confianza
            texto = f"Conf: {puntaje:.2f}"
            y_text = max(0, caja[1] - 20)
            draw.rectangle([caja[0], y_text, caja[0] + 75, y_text + 20], fill="#00FF00")
            
            # Texto negro centrado en el fondo
            draw.text((caja[0] + 2, y_text + 2), texto, fill="black")
    else:
        print("⚠️ No hay cajas detectadas para dibujar.")
            
    return img_dibujada

In [2]:
from pathlib import Path

def load_all_videos(folder_path: str):

    video_folder_path = Path(folder_path)
    videos_path_list = video_folder_path.rglob("*.mov")

    return list(videos_path_list)

In [ ]:
from transformers import Sam3VideoModel, Sam3VideoProcessor
from transformers.video_utils import load_video
from accelerate import Accelerator
import random
import os
from time import sleep
import torch
import gc
import numpy
from PIL import Image
import cv2
from tqdm import tqdm

torch.cuda.empty_cache()
gc.collect()
device = "cuda" if torch.cuda.is_available() else "cpu"

# Extraer un solo video de la lista
random.seed(42) # Con esto se controla que video sale, va a ser uno aleatorio
videos = load_all_videos("Videos/")
video = random.choice(videos)

# Hace que se vea el video, abre una ventana
os.startfile(video)

#sleep(float(duracion))

# Cargar el modelo
device = Accelerator().device
model = Sam3VideoModel.from_pretrained("facebook/sam3").to(device, dtype=torch.bfloat16)
processor = Sam3VideoProcessor.from_pretrained("facebook/sam3")

# --- Cargar los frames por lotes ---
frames_por_lote = 100
frames_chunk = []
num_bloque = 0
ultimo_box = None # Aquí guardamos la ubicación de la pelota entre bloques
outputs_totales = {}
global_frame_idx = 0 

# Prompt
prompt = "orange ball"

# Cargar el video con opencv
cap = cv2.VideoCapture(str(video))

# --- NUEVO: Configurar el archivo de video de salida ---
video_salida = "resultado_sam3.mp4"
fps = cap.get(cv2.CAP_PROP_FPS)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(video_salida, fourcc, fps, (frame_width, frame_height))
# -------------------------------------------------------

# Poner barra que dice cuanto va a durar
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
barra_progreso = tqdm(total=total_frames, desc="Procesando IA", unit="frame")

while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    # Convertir colores de OpenCV (BGR) al estándar que usa la IA (RGB)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frames_chunk.append(frame_rgb)

    if len(frames_chunk) == frames_por_lote:
        # Imprime correctamente el rango de frames
        print(f"Procesando bloque {num_bloque} (Frames {global_frame_idx} a {global_frame_idx + frames_por_lote - 1})...")
        
        # Iniciar sesión SOLO para estos 100 frames
        inference_session = processor.init_video_session(
            video=frames_chunk,
            inference_device=device,
            processing_device="cuda",
            video_storage_device="cpu", # Mantiene tu RAM a salvo
            dtype=torch.bfloat16,
        )
        
        inference_session = processor.add_text_prompt(
            inference_session=inference_session, 
            text="orange ball"
        )
            
        # Propagar la máscara en estos 100 frames
        local_idx = 0 # <-- NUEVO: Índice para saber qué frame dibujar
        for model_outputs in model.propagate_in_video_iterator(inference_session):
            processed_outputs = processor.postprocess_outputs(inference_session, model_outputs)
            
            outputs_totales[global_frame_idx] = {
                "boxes": processed_outputs.get("boxes", []),
                "scores": processed_outputs.get("scores", []),
                "object_ids": processed_outputs.get("object_ids", [])
            }
            
            # --- NUEVO: Dibujar y grabar en el video ---
            # Recuperamos el frame actual y lo regresamos a BGR (los colores de OpenCV)
            frame_dibujo = cv2.cvtColor(frames_chunk[local_idx], cv2.COLOR_RGB2BGR)
            
            # Dibujar máscara translúcida
            if "masks" in processed_outputs and len(processed_outputs["masks"]) > 0:
                mask = processed_outputs["masks"][0].cpu().numpy()
                if len(mask.shape) == 3: mask = mask[0]
                mascara_color = numpy.zeros_like(frame_dibujo)
                mascara_color[mask > 0.5] = [0, 255, 0]
                frame_dibujo = cv2.addWeighted(frame_dibujo, 1.0, mascara_color, 0.4, 0)

            # Dibujar los rectángulos
            boxes = processed_outputs.get("boxes", [])
            scores = processed_outputs.get("scores", [])
            if len(boxes) > 0:
                # Usamos zip para emparejar cada caja con su calificación correspondiente
                for box, score in zip(boxes, scores):
                    x_min, y_min, x_max, y_max = map(int, box)
                    
                    alto = y_max - y_min
                    ancho = x_max - x_min
                    
                    # Convertimos el score a float por si viene como un tensor de PyTorch
                    if float(score) >= 0.75 and alto <= 300 and ancho <= 300:
                        cv2.rectangle(frame_dibujo, (x_min, y_min), (x_max, y_max), (0, 255, 0), 3)
            
            # Guardamos el frame ya dibujado en el archivo mp4
            out.write(frame_dibujo)
            local_idx += 1
            # ------------------------------------------
            
            global_frame_idx += 1

            barra_progreso.update(1)
        
        frames_chunk = []
        num_bloque += 1
        
        del inference_session
        gc.collect() 
        torch.cuda.empty_cache()

cap.release()
out.release()
barra_progreso.close()
print(f"¡Video completo! Se procesaron {len(outputs_totales)} frames exitosamente. Guardado como '{video_salida}'")

C:\Users\Oscar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\triton\windows_utils.py:440: UserWarning: Failed to find CUDA.
  warnings.warn("Failed to find CUDA.")


Loading weights:   0%|          | 0/1797 [00:00<?, ?it/s]

Procesando IA:   0%|          | 0/2716 [00:00<?, ?frame/s]

Procesando bloque 0 (Frames 0 a 99)...


[transformers] Failed to load cv_utils kernel (your torch/cuda setup may not be supported): Kernel repository 'kernels-community/cv-utils' could not verify publisher trust status. Set trust_remote_code=True to allow loading kernels from untrusted sources.. NMS post-processing, hole filling, and sprinkle removal will be skipped.
Procesando IA:   4%|▎         | 96/2716 [00:59<04:44,  9.21frame/s]  

...

Procesando IA:  96%|█████████▌| 2596/2716 [26:15<00:18,  6.48frame/s]

Procesando bloque 26 (Frames 2600 a 2699)...


Procesando IA:  99%|█████████▉| 2700/2716 [27:24<00:09,  1.64frame/s]


¡Video completo! Se procesaron 2700 frames exitosamente. Guardado como 'resultado_sam3.mp4'


KeyError: 'masks'